# 09a - wav2vec2 Embedding 萃取 (Colab GPU)

從 fold-1 best checkpoint 萃取 wav2vec2 最後一層 hidden state 的 mean pooling embedding。

- **輸入**：`audio_16k/*.wav` + `wav2vec_fold1_best.pt`
- **輸出**：`data/embeddings/wav2vec_embeddings.npy` — shape (11318, 768)
- **預估時間**：~5-10 min on T4 GPU

In [ ]:
# === 安裝相依套件 ===
!pip install -q transformers librosa soundfile

In [ ]:
# === 路徑設定（支援 Colab 與本地執行）===
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/SER-Project')
except (ImportError, ModuleNotFoundError):
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

AUDIO_DIR = PROJECT_ROOT / 'data' / 'processed' / 'audio_16k'
CKPT_PATH = PROJECT_ROOT / 'models' / 'checkpoints' / 'wav2vec' / 'wav2vec_fold1_best.pt'
EMB_DIR = PROJECT_ROOT / 'data' / 'embeddings'
EMB_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
import torch
print(f'Device: {"GPU (" + torch.cuda.get_device_name(0) + ")" if torch.cuda.is_available() else "CPU"}')

In [ ]:
# === 複製音訊到 Colab 本地暫存 ===
import shutil

LOCAL_AUDIO = Path('/content/temp_audio')

if LOCAL_AUDIO.exists() and len(list(LOCAL_AUDIO.glob('*.wav'))) > 11000:
    print(f'Local cache exists: {len(list(LOCAL_AUDIO.glob("*.wav"))):,} files')
else:
    print('Copying audio files to local storage (~1-2 min)...')
    LOCAL_AUDIO.mkdir(exist_ok=True)
    !cp "{AUDIO_DIR}"/*.wav /content/temp_audio/
    n_files = len(list(LOCAL_AUDIO.glob('*.wav')))
    print(f'Done: {n_files:,} files copied')

In [ ]:
# === Imports + 設定 ===
import numpy as np
import pandas as pd
import torch
import librosa
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor
from tqdm.auto import tqdm

TARGET_SR = 16000
MAX_LENGTH_SAMPLES = 48000  # 3 sec
BATCH_SIZE = 16
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device: {DEVICE}')

In [ ]:
# === Dataset ===
class AudioDataset(Dataset):
    def __init__(self, paths, feature_extractor, max_length=MAX_LENGTH_SAMPLES):
        self.paths = paths
        self.fe = feature_extractor
        self.max_length = max_length

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        audio, _ = librosa.load(self.paths[idx], sr=TARGET_SR)
        if len(audio) > self.max_length:
            start = (len(audio) - self.max_length) // 2
            audio = audio[start:start + self.max_length]
        elif len(audio) < self.max_length:
            pad_total = self.max_length - len(audio)
            audio = np.pad(audio, (pad_total // 2, pad_total - pad_total // 2))
        inputs = self.fe(audio, sampling_rate=TARGET_SR, return_tensors='pt', padding=False)
        return inputs['input_values'].squeeze(0)


# === 載入 metadata ===
df = pd.read_csv(PROJECT_ROOT / 'data' / 'metadata.csv')
audio_paths = [str(LOCAL_AUDIO / Path(p).name) for p in df['processed_path']]
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained('facebook/wav2vec2-base')

ds = AudioDataset(audio_paths, feature_extractor)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Samples: {len(ds):,}')

In [ ]:
# === 載入 fold-1 best model ===
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    'facebook/wav2vec2-base',
    num_labels=6,
    classifier_proj_size=256,
)
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True))
model.to(DEVICE)
model.eval()

print('Model loaded.')

In [ ]:
# === 萃取 embedding: last hidden state mean pooling → (N, 768) ===
OUT_PATH = EMB_DIR / 'wav2vec_embeddings.npy'

if OUT_PATH.exists():
    existing = np.load(OUT_PATH)
    print(f'[SKIP] {OUT_PATH.name} already exists — shape {existing.shape}')
else:
    all_emb = []

    with torch.no_grad():
        for X in tqdm(loader, desc='wav2vec embedding'):
            X = X.to(DEVICE)
            with torch.cuda.amp.autocast():
                outputs = model.wav2vec2(X, output_hidden_states=True)
            last_hidden = outputs.hidden_states[-1]  # (batch, seq_len, 768)
            emb = last_hidden.mean(dim=1)            # (batch, 768)
            all_emb.append(emb.cpu().float().numpy())

    embeddings = np.concatenate(all_emb, axis=0)
    np.save(OUT_PATH, embeddings)
    print(f'Saved: {OUT_PATH} — shape {embeddings.shape}')

In [ ]:
# === 驗證 ===
emb = np.load(EMB_DIR / 'wav2vec_embeddings.npy')
print(f'Shape: {emb.shape}')       # 預期 (11318, 768)
print(f'dtype: {emb.dtype}')       # 預期 float32
print(f'NaN count: {np.isnan(emb).sum()}')
print(f'Range: [{emb.min():.4f}, {emb.max():.4f}]')
print(f'\nDone! 請將 data/embeddings/wav2vec_embeddings.npy 下載回本地。')